# Scout 02 — Validate coverage
Proves the data is complete BEFORE we model it. Output = your article's
"known limitations" section.

In [ ]:
PROJECT_ID = "statsbomb-football-iq"
from google.colab import auth; auth.authenticate_user()
from google.cloud import bigquery; bq = bigquery.Client(project=PROJECT_ID)
def q(sql): return bq.query(sql.replace("PID",PROJECT_ID)).to_dataframe()

In [ ]:
checks = {
 "matches loaded":   "SELECT COUNT(DISTINCT match_id) n FROM `PID.statsbomb_raw.events_raw`",
 "teams loaded":     "SELECT COUNT(DISTINCT team) n FROM `PID.statsbomb_raw.events_raw`",
 "total events":     "SELECT COUNT(*) n FROM `PID.statsbomb_raw.events_raw`",
 "shots w/o xG":     "SELECT COUNTIF(shot_statsbomb_xg IS NULL AND type='Shot') n FROM `PID.statsbomb_raw.events_raw`",
 "passes w/o location":"SELECT COUNTIF(location IS NULL AND type='Pass') n FROM `PID.statsbomb_raw.events_raw`",
}
import pandas as pd
rows=[]
for name,sql in checks.items():
    try: rows.append((name, int(q(sql)["n"][0])))
    except Exception as e: rows.append((name, f"ERR {e}"))
pd.DataFrame(rows, columns=["check","value"])

In [ ]:
# Events per match — should sit around 3,000-4,000
q('''SELECT match_id, COUNT(*) events
     FROM `PID.statsbomb_raw.events_raw` GROUP BY 1 ORDER BY 2''').describe()

**Expect:** 51 matches, 24 teams, ~3,400 events/match. Shots-without-xG and
passes-without-location should be near zero. Anything odd goes in limitations.